# 🚦 VisionAI — YOLOv8 Traffic Model Fine-Tuning

**Fine-tune YOLOv8 on VisDrone (CCTV/road perspective) — runs on Google Colab free T4 GPU**

### ⚡ Before running:
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run cells **top to bottom** with Shift+Enter

### 💾 Disk: Everything on Colab (~107 GB free). Your laptop uses 0 MB during training.

In [ ]:
# CELL 1: Check GPU
import torch, shutil

if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    raise SystemExit('❌ No GPU — Runtime → Change runtime type → T4 GPU')

total, used, free = shutil.disk_usage('/')
print(f'💾 Colab disk: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total')

In [ ]:
# CELL 2: Install ultralytics (latest — fixes PyTorch 2.6 issues)
!pip install ultralytics --upgrade -q
import ultralytics
print(f'✅ Ultralytics version: {ultralytics.__version__}')
ultralytics.checks()

In [ ]:
# CELL 3: Mount Google Drive (model will be saved here permanently)
from google.colab import drive
import os
drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/VisionAI_Models'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Models will be saved to: {SAVE_DIR}')

In [ ]:
# CELL 4: Set ultralytics dataset directory
# This tells ultralytics WHERE to download and look for datasets
from ultralytics import settings as ult_settings
import os

DATASET_BASE = '/content/datasets'
os.makedirs(DATASET_BASE, exist_ok=True)
ult_settings.update({'datasets_dir': DATASET_BASE})

print(f'✅ Ultralytics dataset dir: {DATASET_BASE}')
print('   VisDrone will auto-download here when training starts')
print(f'   Current ultralytics settings: {dict(ult_settings)}')

In [ ]:
# CELL 5: Fine-tune YOLOv8 on VisDrone — FULLY FIXED
# Fixes: PyTorch 2.6 weights_only + dataset path
import torch, functools, os

# ── Fix: PyTorch 2.6 weights_only=True breaks ultralytics ────
_orig = torch.load
@functools.wraps(_orig)
def _safe_load(f, *a, **kw):
    kw.setdefault('weights_only', False)
    return _orig(f, *a, **kw)
torch.load = _safe_load
print('✅ PyTorch 2.6 patch applied')
# ─────────────────────────────────────────────────────────────

from ultralytics import YOLO

BASE_MODEL = 'yolov8n'   # change to yolov8s for +5% mAP
EPOCHS     = 50
BATCH      = 16
IMG_SIZE   = 640

print(f'🚀 Starting: {BASE_MODEL} × {EPOCHS} epochs on VisDrone')
print(f'   ~2-3 hours on T4 GPU — keep this tab open!\n')

model = YOLO(f'{BASE_MODEL}.pt')

results = model.train(
    data='VisDrone.yaml',      # ultralytics built-in, auto-downloads dataset
    epochs=EPOCHS,
    batch=BATCH,
    imgsz=IMG_SIZE,
    device='cuda',
    project='/content/runs',
    name=f'{BASE_MODEL}_visdrone',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3,
    patience=15,
    mosaic=1.0,
    mixup=0.1,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    scale=0.5,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
)

print('\n🎉 Training complete!')

In [ ]:
# CELL 6: Show training metrics + charts
from IPython.display import Image as IPImage, display
import glob, os

run_dir = f'/content/runs/{BASE_MODEL}_visdrone'
metrics = results.results_dict

print(f'{'='*50}')
print('  TRAINING METRICS')
print(f'{'='*50}')
print(f"  mAP@0.5      : {metrics.get('metrics/mAP50(B)',0)*100:.2f}%")
print(f"  mAP@0.5:0.95 : {metrics.get('metrics/mAP50-95(B)',0)*100:.2f}%")
print(f"  Precision    : {metrics.get('metrics/precision(B)',0)*100:.2f}%")
print(f"  Recall       : {metrics.get('metrics/recall(B)',0)*100:.2f}%")
print(f'{'='*50}')

for chart in glob.glob(os.path.join(run_dir, '*.png'))[:6]:
    print(f'\n📈 {os.path.basename(chart)}')
    display(IPImage(chart, width=700))

In [ ]:
# CELL 7: Validate + generate Confusion Matrix, PR Curve, F1 Curve
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import os

best_path = f'/content/runs/{BASE_MODEL}_visdrone/weights/best.pt'
best_model = YOLO(best_path)

val_r = best_model.val(data='VisDrone.yaml', conf=0.25, iou=0.45, plots=True, verbose=True)

print(f'\nmAP@0.5: {val_r.box.map50*100:.2f}%  |  Precision: {val_r.box.mp*100:.2f}%  |  Recall: {val_r.box.mr*100:.2f}%')

val_dir = f'/content/runs/{BASE_MODEL}_visdrone/val'
for name in ['confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    p = os.path.join(val_dir, name)
    if os.path.exists(p):
        print(f'\n📊 {name}')
        display(IPImage(p, width=700))

In [ ]:
# CELL 8: Save to Google Drive + download to laptop
import shutil, os
from google.colab import files
from ultralytics import YOLO

best_path = f'/content/runs/{BASE_MODEL}_visdrone/weights/best.pt'
size_mb   = os.path.getsize(best_path) / 1e6

# Save to Drive
drive_dest = f'{SAVE_DIR}/{BASE_MODEL}_visdrone_best.pt'
shutil.copy(best_path, drive_dest)
print(f'✅ Saved to Google Drive: {drive_dest} ({size_mb:.1f} MB)')

# Export ONNX
try:
    YOLO(best_path).export(format='onnx', dynamic=True, simplify=True)
    onnx = best_path.replace('.pt', '.onnx')
    shutil.copy(onnx, f'{SAVE_DIR}/{BASE_MODEL}_visdrone_best.onnx')
    print('✅ ONNX exported to Drive')
except Exception as e:
    print(f'ONNX skipped: {e}')

# Download to laptop (~6MB only!)
print(f'\n⬇️  Downloading best.pt ({size_mb:.1f} MB) to your laptop...')
files.download(best_path)

print('\n✅ Done! Next steps:')
print(f'  1. Move best.pt → real_time_object detection/models/{BASE_MODEL}_visdrone_best.pt')
print(f'  2. Edit .env: MODEL_SIZE={BASE_MODEL}_visdrone_best')
print('  3. Restart backend — your traffic model is live! 🎯')

In [ ]:
# CELL 9: Test on a sample road image
from ultralytics import YOLO
from IPython.display import Image as IPImage, display
import urllib.request, os

url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/e/e7/Phuket_road.jpg/1280px-Phuket_road.jpg'
urllib.request.urlretrieve(url, '/content/road_test.jpg')

best_path = f'/content/runs/{BASE_MODEL}_visdrone/weights/best.pt'
m = YOLO(best_path)
res = m.predict('/content/road_test.jpg', conf=0.25, save=True, project='/content', name='road_test')

print('📊 Detections:')
for box in res[0].boxes:
    print(f"   {res[0].names[int(box.cls[0])]:20s}  {float(box.conf[0])*100:.1f}%")

out = '/content/road_test/road_test.jpg'
if os.path.exists(out):
    display(IPImage(out, width=800))
    print('\n✅ Fine-tuned model working!')